# PHASE 2: DATA CLEANING & PREPROCESSING PIPELINE
**Học phần:** DTA301 – Data Analytics  
**Người thực hiện:** Lê Đỗ Nhật Anh (SE180055)  
**Mục tiêu:**
- Tải và làm sạch duy nhất tập dữ liệu gốc **SurveilDrone-Net23.csv** (140,256 bản ghi).
- Kiểm tra trùng lặp (duplicates), xử lý missing values (nếu có) và áp dụng ràng buộc vật lý.
- Chuẩn hóa định dạng thời gian (`timestamp`) và các trường phân loại.
- Tạo nhãn an toàn bay (`unstable_flight`) chuẩn quy chuẩn kỹ thuật UAV.
- Xuất dữ liệu sạch vào thư mục `data/processed/SurveilDrone_Net23_cleaned.csv`.

In [5]:
import os
import sys
import random
import numpy as np
import pandas as pd

# 1. Cố định tính tái lập (Reproducibility)
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 2. Danh sách các đường dẫn có thể chứa file dữ liệu gốc
possible_paths = [
    "SurveilDrone-Net23.csv",
    "data/SurveilDrone-Net23.csv",
    "../data/SurveilDrone-Net23.csv",
    "../data/raw/SurveilDrone-Net23.csv",
    "../../data/SurveilDrone-Net23.csv"
]

surveil_raw_path = None
for path in possible_paths:
    if os.path.exists(path):
        surveil_raw_path = path
        break

if surveil_raw_path is None:
    raise FileNotFoundError(
        "❌ KHÔNG TÌM THẤY FILE 'SurveilDrone-Net23.csv'. "
        "Hãy đảm bảo bạn đã copy file SurveilDrone-Net23.csv vào thư mục dự án hoặc thư mục data/!"
    )

print(f"--- BẮT ĐẦU NẠP VÀ LÀM SẠCH TỪ: {surveil_raw_path} ---")
df_surveil = pd.read_csv(surveil_raw_path)
print(f"Kích thước ban đầu: {df_surveil.shape}")

# 3. Loại bỏ các dòng trùng lặp (duplicates)
df_surveil = df_surveil.drop_duplicates()

# 4. Kiểm tra và xử lý Missing Values
missing_counts = df_surveil.isnull().sum()
if missing_counts.sum() > 0:
    print("\nPhát hiện Missing Values -> Tiến hành nội suy/điền...")
    df_surveil = df_surveil.interpolate(method='linear').bfill().ffill()
else:
    print("\n✅ Không phát hiện giá trị khuyết thiếu (Missing values = 0).")

# 5. Ràng buộc giới hạn vật lý
if 'battery_level_pct' in df_surveil.columns:
    df_surveil = df_surveil[(df_surveil['battery_level_pct'] >= 0) & (df_surveil['battery_level_pct'] <= 100)]

if 'wind_speed_mps' in df_surveil.columns:
    df_surveil = df_surveil[(df_surveil['wind_speed_mps'] >= 0) & (df_surveil['wind_speed_mps'] <= 35)]

# 6. Chuẩn hóa DateTime
if 'timestamp' in df_surveil.columns:
    df_surveil['timestamp'] = pd.to_datetime(df_surveil['timestamp'])

# 7. Gán nhãn An toàn bay (unstable_flight) chuẩn quy chuẩn UAV
df_surveil['unstable_flight'] = (
    (df_surveil['battery_level_pct'] < 20) | 
    (df_surveil['wind_speed_mps'] > 8.0) | 
    (df_surveil['proximity_to_restricted_zone_m'] < 50)
).astype(int)

# 8. Kiểm tra kết quả sau khi làm sạch
print("\n--- KIỂM TRA DỮ LIỆU SAU KHI LÀM SẠCH ---")
print(f"Kích thước dữ liệu sạch: {df_surveil.shape}")
print(f"Số lượng vụ bay mất an toàn (unstable_flight = 1): {df_surveil['unstable_flight'].sum()} / {len(df_surveil)}")

# 9. Tạo thư mục output và xuất file sạch
output_dir = "data/processed" if os.path.exists("data") else "../data/processed"
os.makedirs(output_dir, exist_ok=True)
processed_path = os.path.join(output_dir, "SurveilDrone_Net23_cleaned.csv")

df_surveil.to_csv(processed_path, index=False)
print(f"\n🎉 THÀNH CÔNG: Đã lưu dữ liệu sạch duy nhất vào `{processed_path}`")

--- BẮT ĐẦU NẠP VÀ LÀM SẠCH TỪ: ../data/SurveilDrone-Net23.csv ---
Kích thước ban đầu: (140256, 34)

✅ Không phát hiện giá trị khuyết thiếu (Missing values = 0).

--- KIỂM TRA DỮ LIỆU SAU KHI LÀM SẠCH ---
Kích thước dữ liệu sạch: (137085, 35)
Số lượng vụ bay mất an toàn (unstable_flight = 1): 45348 / 137085

🎉 THÀNH CÔNG: Đã lưu dữ liệu sạch duy nhất vào `../data/processed\SurveilDrone_Net23_cleaned.csv`
